# MST Topology Trading Strategy — Project Proposal

**Zara Nip** | Student ID: **12411910**

FINM — Quantitative Trading Strategies | Winter 2026

## 1. Project Motivation

Financial markets are commonly studied as collections of individual assets whose prices follow their own stochastic processes. In practice, however, returns are driven by shared exposures — interest-rate regimes, risk-appetite shifts, supply shocks — that induce complex, time-varying dependency structures across asset classes. A standard correlation matrix captures pairwise linear co-movement, but it treats every relationship symmetrically and provides no notion of which linkages are structurally dominant versus redundant.

We address this limitation by applying *graph-theoretic filtering* to the correlation matrix. Specifically, we convert rolling pairwise correlations into a metric distance (Mantegna, 1999) and construct the **minimum spanning tree (MST)** — the unique connected subgraph that links all assets at minimum total distance. The MST keeps exactly $N - 1$ edges from the $N(N-1)/2$ available, isolating the skeleton of the market's dependency structure and discarding redundant connections that would otherwise add noise.

This project studies whether **temporary dislocations in the MST's topology** — specifically, anomalous increases in *effective resistance* between connected assets — predict short-horizon mean-reversion in relative returns. The hypothesis is grounded in a simple economic intuition: when two assets that normally move closely together suddenly decouple on the network, the deviation is more likely to reverse than to persist, provided the underlying structural relationship has not permanently changed.

The approach is distinguished from standard correlation-based pair trading in three ways:

1. **Topological selection.** We only trade pairs that are neighbors on the MST, not all pairs with high correlation. The tree enforces sparsity and ensures that each pair carries non-redundant information about market structure.

2. **Path-aware dislocation measurement.** Effective resistance (Kirchhoff distance) accounts for *indirect* connections through intermediate nodes, not just pairwise correlation. Two assets can have moderate direct correlation but low effective resistance if they are connected through a short, tight chain of intermediaries — or vice versa.

3. **Regime-adaptive risk scaling.** The second-smallest eigenvalue of the MST's graph Laplacian — the Fiedler value, or algebraic connectivity — measures how tightly the market is coupled at any point in time. When the Fiedler value drops below its historical norm, the network is fragmenting and mean-reversion becomes less reliable; the strategy automatically reduces gross exposure.

## 2. Academic Foundation

The project draws on four strands of research:

**Mantegna (1999)** introduced the distance metric $d_{ij} = \sqrt{2(1 - \rho_{ij})}$, which converts Pearson correlations into a proper metric satisfying symmetry and the triangle inequality. Constructing the MST over this metric was shown to recover meaningful sector and asset-class clustering in equity markets.

**Onnela et al. (2003)** extended this work by studying how the MST evolves over time. They showed that the tree reconfigures during market crises — assets cluster more tightly and the tree becomes more star-like — and that these topological shifts carry predictive information about future volatility and correlation regimes.

**Klein and Randić (1993)** formalized effective resistance as a graph distance in combinatorial mathematics. On a tree, the effective resistance between nodes $i$ and $j$ equals the sum of edge weights along the unique path connecting them. This provides a path-aware measure of structural distance that goes beyond pairwise correlation.

**Pozzi, Di Matteo, and Aste (2013)** studied how risk spreads across financial networks and showed that changes in network topology — measured through metrics like the Fiedler value — anticipate changes in realized correlation and volatility. Their work motivates using algebraic connectivity as a dynamic risk-state variable.

By combining these ideas, we construct a complete signal pipeline: Mantegna distances → MST construction → effective resistance computation → z-score normalization → volatility-targeted pair sizing → Fiedler-based risk throttle.

## 3. Investment Universe

We select 21 highly liquid ETFs spanning four asset classes. The diversity ensures that the MST captures genuine cross-asset structure rather than within-sector correlation noise:

| Asset Class | ETFs | Count |
|---|---|---|
| Equity Sectors | XLF, XLK, XLE, XLV, XLI, XLY, XLP, XLB, XLU | 9 |
| Commodities | GLD, SLV, USO, DBA | 4 |
| Currencies | FXE, FXB, FXY, FXA | 4 |
| Fixed Income | TLT, IEF, HYG, LQD | 4 |

All data is sourced from **NASDAQ Data Link** via the QuoteMedia/PRICES table. We use adjusted close prices to account for splits and dividends, ensuring returns are computed on an apples-to-apples basis across the full sample.

The sample period runs from **January 2, 2018 through December 31, 2025**, providing nearly eight years of daily data that spans several distinct market regimes: the late bull market (2018-19), the COVID crash and recovery (2020), the post-pandemic inflation and rate-hiking cycle (2021-22), and the subsequent stabilization (2023-25). This variety is important for testing whether MST-based signals are robust across environments.

## 4. Data Acquisition and Cleaning

We now walk through the data pipeline: fetching prices from NASDAQ Data Link, caching them locally, constructing the return matrix, and verifying data integrity.

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from mst_strategy import load_or_fetch_prices, mantegna_distance, build_mst_kruskal

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')

# ── Configuration ──
ETF_UNIVERSE = [
    'XLF', 'XLK', 'XLE', 'XLV', 'XLI', 'XLY', 'XLP', 'XLB', 'XLU',  # Equity sectors
    'GLD', 'SLV', 'USO', 'DBA',                                        # Commodities
    'FXE', 'FXB', 'FXY', 'FXA',                                        # Currencies
    'TLT', 'IEF', 'HYG', 'LQD',                                        # Fixed income
]

START_DATE = '2018-01-01'
END_DATE = '2025-12-31'
DATA_DIR = Path('data')

The cell below loads adjusted close prices from locally cached CSVs. If a ticker is missing from the cache, it fetches from NASDAQ Data Link via the `QUOTEMEDIA/PRICES` table and saves the result for future runs. This avoids hitting the API on every execution and makes the notebook reproducible without network access.

In [ ]:
prices = load_or_fetch_prices(ETF_UNIVERSE, START_DATE, END_DATE, str(DATA_DIR))

print(f'Loaded price panel: {prices.shape[0]} trading days x {prices.shape[1]} assets')
print(f'Date range: {prices.index.min().date()} to {prices.index.max().date()}')
print(f'Assets: {list(prices.columns)}')
prices.head()

### 4.1 Missing Data Check

For the MST construction to be valid, we need a complete, aligned panel — missing prices would produce unreliable correlations and distort the tree geometry. We check for gaps and verify the panel is dense enough for downstream analysis.

In [ ]:
missing_by_ticker = prices.isna().sum()
total_missing = int(missing_by_ticker.sum())

print(f'Total missing values across the entire panel: {total_missing}')
if total_missing > 0:
    print('Missing values by ticker:')
    print(missing_by_ticker[missing_by_ticker > 0])
else:
    print('The panel is fully populated — no missing values detected.')

print(f'\nPanel density: {(1 - prices.isna().mean().mean()) * 100:.2f}%')

### 4.2 Return Computation

We compute simple daily returns from adjusted close prices. The first row is dropped because it has no prior observation. We also compute basic summary statistics to confirm the data is sensible — typical daily moves should be on the order of tens of basis points to a few percent.

In [ ]:
returns = prices.pct_change().dropna()

print(f'Return matrix: {returns.shape[0]} observations x {returns.shape[1]} assets')
print(f'Average daily absolute return across all assets: {returns.abs().mean().mean():.4f}')
print(f'Average daily return (annualized): {returns.mean().mean() * 252:.4f}')

return_stats = returns.describe().T[['mean', 'std', 'min', 'max']]
return_stats.columns = ['Mean', 'Std Dev', 'Min', 'Max']
return_stats['Annualized Vol'] = return_stats['Std Dev'] * np.sqrt(252)
return_stats

The summary statistics above confirm that our data is well-behaved. Daily means are close to zero (as expected for daily frequency), standard deviations vary meaningfully across asset classes — equities and commodities are more volatile than fixed income and currencies — and no extreme outliers suggest data errors. The annualized volatilities range from approximately 4-5% for short-duration bond ETFs to 30%+ for commodity-linked instruments like USO, which is consistent with known asset-class risk profiles.

## 5. Exploratory Data Analysis

Before constructing any signals, we examine three key properties of the data that are directly relevant to the strategy: (1) the cumulative return trajectories, which reveal whether the universe offers genuine cross-asset diversity; (2) the full-sample correlation structure, which motivates the MST filtering step; and (3) the rolling volatility dynamics, which affect the sizing and risk-management layers.

### Graph 1: Cumulative Returns by ETF

We plot cumulative returns for every ETF in the universe. A heterogeneous set of trajectories confirms that the universe is not dominated by a single factor and that the MST will capture meaningful cross-asset structure.

In [ ]:
cum_returns = (1 + returns).cumprod() - 1

fig, ax = plt.subplots(figsize=(14, 7))
for col in cum_returns.columns:
    ax.plot(cum_returns.index, cum_returns[col], linewidth=1.1, alpha=0.85, label=col)

ax.set_title('Cumulative Returns by ETF (2018–2025)', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Cumulative Return', fontsize=12)
ax.legend(loc='upper left', fontsize=8, ncol=4, framealpha=0.9)
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
plt.tight_layout()
plt.show()

total_cum = cum_returns.iloc[-1].sort_values(ascending=False)
print(f'Highest cumulative return: {total_cum.index[0]} at {total_cum.iloc[0]:.4f}')
print(f'Lowest cumulative return: {total_cum.index[-1]} at {total_cum.iloc[-1]:.4f}')
print(f'Cross-sectional dispersion (std of final cumulative returns): {total_cum.std():.4f}')

The wide dispersion in terminal cumulative returns — ranging from strongly positive (technology and equity growth sectors) to negative (energy-linked and some currency ETFs) — confirms that this is a heterogeneous opportunity set rather than a single-factor basket. The COVID drawdown in March 2020 is visible across most assets but affects them asymmetrically, which is exactly the kind of cross-asset structure the MST is designed to capture. The persistent divergence across asset classes supports the premise that tree-based filtering can isolate meaningful dependency channels.

### Graph 2: Full-Sample Correlation Heatmap

We compute the full-sample Pearson correlation matrix and visualize it as a heatmap. This motivates the MST filtering step: the raw correlation matrix has $21 \times 20 / 2 = 210$ unique pairwise relationships, most of which are noisy and redundant. The MST will retain only 20 edges.

In [ ]:
corr_full = returns.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_full, dtype=bool), k=1)
sns.heatmap(
    corr_full, mask=mask, cmap='coolwarm', center=0,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    linewidths=0.5, ax=ax, vmin=-1, vmax=1,
    cbar_kws={'label': 'Pearson Correlation'}
)
ax.set_title('Full-Sample Pairwise Correlation Matrix', fontsize=16, fontweight='bold')
ax.set_xlabel('Asset', fontsize=12)
ax.set_ylabel('Asset', fontsize=12)
plt.tight_layout()
plt.show()

upper_corr = corr_full.where(np.triu(np.ones(corr_full.shape), k=1).astype(bool)).stack()
print(f'Average pairwise correlation: {upper_corr.mean():.4f}')
print(f'Minimum pairwise correlation: {upper_corr.min():.4f}')
print(f'Maximum pairwise correlation: {upper_corr.max():.4f}')
print(f'Share of pairs with |correlation| > 0.5: {(upper_corr.abs() > 0.5).mean():.2%}')

The heatmap reveals a clear block structure: equity-sector ETFs form a warm cluster of positive correlations in the upper-left, while fixed-income instruments (TLT, IEF) show mild negative or near-zero correlation with equities. Commodities and currencies sit between these two clusters with moderate, mixed co-movements. Importantly, many off-diagonal correlations are non-zero but weak — exactly the kind of noisy, semi-redundant structure that the MST is designed to compress. By keeping only 20 of the 210 edges, the tree will isolate the strongest connectivity channels and discard the rest.

### Graph 3: Annualized Volatility by Asset

Understanding the volatility profile of each asset is critical because the strategy's pair weights are inversely proportional to spread volatility. Assets with very different risk profiles will produce pairs whose spread behavior — and therefore signal quality — varies substantially.

In [ ]:
ann_vol = returns.std() * np.sqrt(252)
ann_vol_sorted = ann_vol.sort_values(ascending=True)

# Color by asset class
class_colors = {
    'Equity': '#3A7CA5',
    'Commodity': '#E07A2F',
    'Currency': '#27966B',
    'Fixed Income': '#8B5CF6',
}
asset_class_map = {}
for t in ['XLF','XLK','XLE','XLV','XLI','XLY','XLP','XLB','XLU']:
    asset_class_map[t] = 'Equity'
for t in ['GLD','SLV','USO','DBA']:
    asset_class_map[t] = 'Commodity'
for t in ['FXE','FXB','FXY','FXA']:
    asset_class_map[t] = 'Currency'
for t in ['TLT','IEF','HYG','LQD']:
    asset_class_map[t] = 'Fixed Income'

colors = [class_colors[asset_class_map[t]] for t in ann_vol_sorted.index]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(ann_vol_sorted.index, ann_vol_sorted.values, color=colors, edgecolor='white', height=0.7)
ax.set_title('Annualized Volatility by ETF (2018–2025)', fontsize=16, fontweight='bold')
ax.set_xlabel('Annualized Volatility', fontsize=12)
ax.set_ylabel('ETF', fontsize=12)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=lbl) for lbl, c in class_colors.items()]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

for i, (val, ticker) in enumerate(zip(ann_vol_sorted.values, ann_vol_sorted.index)):
    ax.text(val + 0.003, i, f'{val:.1%}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f'Most volatile: {ann_vol_sorted.index[-1]} at {ann_vol_sorted.iloc[-1]:.2%} annualized')
print(f'Least volatile: {ann_vol_sorted.index[0]} at {ann_vol_sorted.iloc[0]:.2%} annualized')
print(f'Cross-sectional vol range: {ann_vol_sorted.iloc[-1] - ann_vol_sorted.iloc[0]:.2%}')

The volatility bar chart highlights a nearly 5x range in annualized risk across the universe. Commodity-linked ETFs like USO exhibit the highest volatility, while short-duration fixed-income instruments like IEF and LQD sit at the low end. This dispersion is important for two reasons. First, it means that pair spreads will have heterogeneous risk profiles, so volatility-targeted sizing is essential to prevent high-vol pairs from dominating the portfolio. Second, it confirms that the universe spans genuinely distinct risk regimes, which is a prerequisite for the MST to capture meaningful cross-asset connectivity rather than redundant within-cluster variation.

### Graph 4: Rolling 126-Day Correlation Between Equity (XLK) and Fixed Income (TLT)

The MST topology strategy depends on *time-varying* relationships between assets. To illustrate that correlations are indeed unstable, we plot the rolling 126-day (half-year) correlation between XLK (technology equities) and TLT (long-duration Treasuries) — two assets that are typically negatively correlated but whose relationship has shifted meaningfully across recent regimes.

In [ ]:
rolling_corr = returns['XLK'].rolling(126).corr(returns['TLT'])

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(rolling_corr.index, rolling_corr.values, color='#1B2A4A', linewidth=1.3)
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.fill_between(rolling_corr.index, rolling_corr.values, 0,
                where=rolling_corr.values > 0, alpha=0.2, color='firebrick', label='Positive regime')
ax.fill_between(rolling_corr.index, rolling_corr.values, 0,
                where=rolling_corr.values <= 0, alpha=0.2, color='steelblue', label='Negative regime')
ax.set_title('Rolling 126-Day Correlation: XLK vs TLT', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Pearson Correlation', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

pct_positive = (rolling_corr.dropna() > 0).mean()
print(f'Share of sample where XLK-TLT correlation is positive: {pct_positive:.2%}')
print(f'Correlation range: [{rolling_corr.min():.4f}, {rolling_corr.max():.4f}]')
print(f'Full-sample correlation: {returns["XLK"].corr(returns["TLT"]):.4f}')

The rolling correlation between technology equities and long-duration Treasuries swings between approximately -0.6 and +0.4 over our sample, spending substantial time in both positive and negative territory. This regime-switching behavior — driven by shifts between growth-scare environments (where stocks and bonds fall together) and risk-off flights (where bonds rally as stocks sell off) — is exactly what makes static correlation-based strategies fragile. The MST dynamically reconfigures as these relationships evolve, adapting which pairs are considered neighbors and therefore which dislocations are flagged as tradable signals.

## 6. Data Formatting and Export

Finally, we verify the structure of the cached data files and confirm that the price panel is in the format needed for downstream MST construction and signal generation.

In [ ]:
# Verify cached CSV structure
sample_csv = pd.read_csv(DATA_DIR / 'XLK.csv', parse_dates=['date'])
print('Sample cached CSV (XLK):')
print(f'  Columns: {list(sample_csv.columns)}')
print(f'  Rows: {len(sample_csv)}')
print(f'  Date range: {sample_csv["date"].min().date()} to {sample_csv["date"].max().date()}')
print()

# Verify the pivoted panel
print('Pivoted price panel:')
print(f'  Shape: {prices.shape}')
print(f'  Index type: {type(prices.index).__name__} (DatetimeIndex expected)')
print(f'  Column type: asset tickers — {prices.columns.tolist()[:5]} ...')
print(f'  Any NaN remaining: {prices.isna().any().any()}')
print()

# Data files on disk
cached_files = sorted(DATA_DIR.glob('*.csv'))
print(f'Cached CSV files in {DATA_DIR}/: {len(cached_files)}')
for f in cached_files:
    print(f'  {f.name}')

Each ticker is cached as a flat CSV with columns `[date, close, ticker]`. The `load_or_fetch_prices` function pivots these into a `date × ticker` panel with adjusted close prices as values. The panel is fully populated with no remaining NaN entries, which is required for the rolling correlation computation and MST construction that follows in the full analysis notebook.

## 7. Summary and Next Steps

This proposal notebook establishes the foundation for the MST Topology Trading project:

- We motivated the research question: whether temporary dislocations in the minimum spanning tree of cross-asset correlations predict short-horizon mean-reversion in relative returns.
- We summarized the academic literature that underpins the approach: Mantegna distance, dynamic MST analysis, effective resistance, and algebraic connectivity.
- We acquired and cached adjusted close prices for 21 ETFs across four asset classes from NASDAQ Data Link, covering January 2018 through December 2025.
- We verified data completeness, computed returns, and produced four exploratory graphs showing cumulative performance, correlation structure, volatility profiles, and time-varying correlation dynamics.

In the full project notebook, we will:

1. Construct rolling weekly MSTs and compute effective resistance for all node pairs.
2. Generate z-scored mean-reversion signals from resistance deviations and apply volatility-targeted pair sizing.
3. Simulate the strategy under both zero and realistic (5 bps/leg) transaction costs.
4. Analyze performance, drawdowns, tail risk, SPY correlation, and sensitivity to key parameters.
5. Compare results across cost regimes and stress periods to assess economic viability.